# 02 — Metric & Seasonal-Naive Baseline
### Project FORESIGHT — NorthBay Living

Per the engagement methodology: fix the metric and build the baseline **before** touching a
real model. This notebook defines WAPE, builds the seasonal-naive baseline, and reports its
accuracy — this is the bar `03_model.ipynb` must beat.

In [1]:
import sys
sys.path.append('../src')
import pandas as pd
import numpy as np
from forecast import wape, bias  # reuse the exact metric functions used in production

weekly = pd.read_csv('../data/processed/weekly_features.csv', parse_dates=['week_start'])
weekly = weekly.sort_values(['sku_id', 'week_start'])
weekly.head()

,sku_id,week_start,units_sold,revenue,avg_price,promo_days,holiday_days,category,subcategory,unit_cost,...,lag_4,lag_8,roll_mean_4,roll_mean_8,roll_std_4,seasonal_naive,on_hand_units,on_order_units,lead_time_days,reorder_point
3866,SKU0001,2025-04-01,106.0,821016.64,7745.44,0,0,Furniture,Shelving,3462.67,...,NaN,NaN,NaN,NaN,NaN,NaN,457,381,25.0,683
4058,SKU0001,2025-04-08,113.0,875234.72,7745.44,0,0,Furniture,Shelving,3462.67,...,NaN,NaN,NaN,NaN,NaN,NaN,306,381,25.0,683
4148,SKU0001,2025-04-15,104.0,805525.76,7745.44,0,0,Furniture,Shelving,3462.67,...,NaN,NaN,NaN,NaN,NaN,NaN,193,771,25.0,683
4243,SKU0001,2025-04-22,137.0,1061125.28,7745.44,0,0,Furniture,Shelving,3462.67,...,NaN,NaN,NaN,NaN,NaN,NaN,470,390,25.0,683
4449,SKU0001,2025-04-29,112.0,867489.28,7745.44,0,0,Furniture,Shelving,3462.67,...,106.0,NaN,115.0,NaN,15.165751,115.0,333,390,25.0,683


## 1. The metric: WAPE

**WAPE (Weighted Absolute Percentage Error)** = total absolute error / total actual demand.

It's the primary accuracy metric for this engagement because it stays well-behaved for
low-volume SKUs, where MAPE explodes (dividing by a near-zero actual).

In [2]:
def wape_demo(y_true, y_pred):
    y_true, y_pred = np.asarray(y_true, float), np.asarray(y_pred, float)
    return np.sum(np.abs(y_true - y_pred)) / np.sum(np.abs(y_true))

# sanity check against the production function
demo = wape_demo([10, 20, 5], [8, 22, 6])
prod = wape([10, 20, 5], [8, 22, 6])
print('WAPE (demo):', round(demo, 4), '| WAPE (production function):', round(prod, 4))

WAPE (demo): 0.1429 | WAPE (production function): 0.1429


## 2. The baseline: seasonal-naive

Predict this week's demand as **the same SKU's demand 52 weeks ago**. This is already
computed in the pipeline as the `seasonal_naive` column (with a rolling-mean fallback for SKUs
without a full year of history yet — see `src/pipeline.py`).

In [3]:
has_baseline = weekly.dropna(subset=['seasonal_naive'])
baseline_wape = wape(has_baseline['units_sold'], has_baseline['seasonal_naive'])
baseline_bias = bias(has_baseline['units_sold'], has_baseline['seasonal_naive'])

print(f"Seasonal-naive baseline — full-history WAPE: {baseline_wape:.4f}")
print(f"Seasonal-naive baseline — bias: {baseline_bias:+.3f} units/week (negative = under-forecasts)")
print(f"Evaluated on {len(has_baseline):,} SKU-week rows")

Seasonal-naive baseline — full-history WAPE: 0.2472
Seasonal-naive baseline — bias: +6.105 units/week (negative = under-forecasts)
Evaluated on 9,869 SKU-week rows


## 3. Per-category baseline accuracy

Useful to see whether the baseline already works well for some categories and poorly for
others — this tells us where the real model (`03_model.ipynb`) needs to work hardest.

In [4]:
cat_wape = (
    has_baseline.groupby('category')
    .apply(lambda g: wape(g['units_sold'], g['seasonal_naive']))
    .sort_values()
)
cat_wape.rename('baseline_WAPE')

category
Decor               0.230483
Furniture           0.240975
Small Appliances    0.241078
Kitchen             0.241994
Bedding             0.280574
Name: baseline_WAPE, dtype: float64

## Takeaway

The seasonal-naive baseline scores **~0.24 WAPE** on backtest (see `03_model.ipynb` for the
fair, rolling-origin comparison — the number here uses full history and is only for
understanding the baseline itself, not the honest evaluation). Any model built next must beat
this on unseen data, not just fit the training period.